# Road Edge Detection
Detect the road surface using SegFormer and draw boundary lines at the edges.

In [ ]:
!pip install transformers torch torchvision pillow opencv-python-headless matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

## 1. Load SegFormer (pre-trained on Cityscapes)
Cityscapes classes include: road, sidewalk, building, wall, fence, pole, traffic light, traffic sign, vegetation, terrain, sky, person, car, truck, bus, train, motorcycle, bicycle.

In [ ]:
model_name = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
processor = SegformerImageProcessor.from_pretrained(model_name)
seg_model = SegformerForSemanticSegmentation.from_pretrained(model_name)
seg_model.eval()
print("Model loaded!")

## 2. Segment the road

In [ ]:
IMAGE_PATH = "images.jpeg"  # <-- change this

img_pil = Image.open(IMAGE_PATH).convert("RGB")
img_cv = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
h, w = img_cv.shape[:2]

# Run segmentation
inputs = processor(images=img_pil, return_tensors="pt")
with torch.no_grad():
    outputs = seg_model(**inputs)

# Get class predictions per pixel
logits = outputs.logits  # (1, num_classes, H/4, W/4)
pred = logits.argmax(dim=1).squeeze().cpu().numpy()

# Resize prediction to original image size
pred_resized = cv2.resize(pred.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)

# Cityscapes class 0 = road, class 1 = sidewalk
road_mask = (pred_resized == 0).astype(np.uint8) * 255

print(f"Image: {w}x{h}")
print(f"Road pixels: {np.sum(road_mask > 0)} / {w*h} ({np.sum(road_mask > 0)*100/(w*h):.1f}%)")

## 3. Show the raw road mask

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(img_rgb)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(road_mask, cmap="gray")
axes[1].set_title("Road Mask")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Find road edges and draw boundary lines

In [ ]:
def detect_road_edges(road_mask, min_contour_area=500):
    """
    Find the edges of the road from the segmentation mask.
    Returns contours and a simplified polygon version.
    """
    # Clean up mask — remove noise
    kernel = np.ones((7, 7), np.uint8)
    clean_mask = cv2.morphologyEx(road_mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_OPEN, kernel, iterations=2)

    # Find contours (edges of the road)
    contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Filter out tiny contours (noise)
    contours = [c for c in contours if cv2.contourArea(c) > min_contour_area]

    # Sort by area — largest first (main road)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    # Simplify contours into cleaner lines
    simplified = []
    for c in contours:
        epsilon = 0.01 * cv2.arcLength(c, True)  # lower = more detail
        approx = cv2.approxPolyDP(c, epsilon, True)
        simplified.append(approx)

    return contours, simplified, clean_mask


contours, simplified, clean_mask = detect_road_edges(road_mask)
print(f"Found {len(contours)} road region(s)")

## 5. Visualize — road edges on the original image

In [ ]:
def draw_road_edges(img_rgb, contours, simplified, road_mask):
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    # 1) Road overlay
    overlay = img_rgb.copy()
    road_colored = np.zeros_like(overlay)
    road_colored[road_mask > 0] = [0, 100, 255]  # blue tint on road
    blended = cv2.addWeighted(overlay, 0.7, road_colored, 0.3, 0)
    axes[0].imshow(blended)
    axes[0].set_title("Road Surface")
    axes[0].axis("off")

    # 2) Raw contours (detailed edges)
    edge_img = img_rgb.copy()
    cv2.drawContours(edge_img, contours, -1, (255, 50, 50), 3)
    axes[1].imshow(edge_img)
    axes[1].set_title("Road Edges (detailed)")
    axes[1].axis("off")

    # 3) Simplified polygon edges (cleaner lines)
    simple_img = img_rgb.copy()
    cv2.drawContours(simple_img, simplified, -1, (0, 255, 100), 3)
    # Draw corner points
    for s in simplified:
        for point in s:
            x, y = point[0]
            cv2.circle(simple_img, (x, y), 5, (255, 255, 0), -1)
    axes[2].imshow(simple_img)
    axes[2].set_title("Road Edges (simplified)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()


draw_road_edges(img_rgb, contours, simplified, road_mask)

## 6. Full segmentation map (all classes)

In [ ]:
CITYSCAPES_CLASSES = {
    0: "road", 1: "sidewalk", 2: "building", 3: "wall", 4: "fence",
    5: "pole", 6: "traffic light", 7: "traffic sign", 8: "vegetation",
    9: "terrain", 10: "sky", 11: "person", 12: "rider", 13: "car",
    14: "truck", 15: "bus", 16: "train", 17: "motorcycle", 18: "bicycle"
}

# Color map
COLORS = np.random.RandomState(42).randint(0, 255, (19, 3), dtype=np.uint8)
COLORS[0] = [128, 64, 128]    # road — purple
COLORS[1] = [244, 35, 232]    # sidewalk — pink
COLORS[13] = [0, 0, 142]      # car — dark blue

# Full segmentation visualization
seg_colored = COLORS[pred_resized]
blended = cv2.addWeighted(img_rgb, 0.5, seg_colored, 0.5, 0)

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(blended)
ax.set_title("Full Segmentation Map")
ax.axis("off")

# Legend
unique_classes = np.unique(pred_resized)
legend_text = ", ".join([f"{CITYSCAPES_CLASSES.get(c, c)}" for c in unique_classes])
print(f"Detected classes: {legend_text}")
plt.tight_layout()
plt.show()

## Next steps
- Use the road contours to **auto-generate zones** for the traffic signal optimizer
- Combine with YOLO: only count cars that are **on the road surface**
- Find **road direction** from the contour shape to determine N/S/E/W automatically